# LLM Evaluation — Without Description

Predict loan outcomes using an LLM with **structured features only** (no borrower description).
Compare results against the ANN model on the same 100 test samples.

## Setup

In [ ]:
import sys
import pandas as pd
import numpy as np
from tqdm import tqdm

from llm_utils import (
    load_llm_sample, run_ann_on_sample,
    build_system_prompt, build_user_prompt,
    call_llm, parse_llm_response,
    evaluate_predictions, compare_results
)

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
# Change these when you decide on a provider/model
API_PROVIDER = "anthropic"   # "anthropic" or "openai"
MODEL_NAME   = None          # None = use default for provider
API_KEY      = None          # None = read from environment variable

## Load Data & Run ANN

In [ ]:
llm_sample = load_llm_sample()
y_true = llm_sample['loan_status'].values

print(f"Sample size: {len(llm_sample)}")
print(f"Class distribution:\n{llm_sample['loan_status'].value_counts()}")

In [ ]:
ann_probs, ann_preds = run_ann_on_sample(llm_sample)
print(f"ANN predictions ready: {len(ann_preds)} samples")

## LLM Predictions (No Description)

In [ ]:
system_prompt = build_system_prompt()

# Preview the prompt for the first loan
sample_prompt = build_user_prompt(llm_sample.iloc[0], include_desc=False)
print("System prompt:")
print(system_prompt)
print("\n" + "=" * 50)
print("\nSample user prompt:")
print(sample_prompt)

In [ ]:
# Run LLM on all 100 samples
llm_predictions = []
llm_reasonings = []
llm_raw_responses = []

for i, (_, row) in enumerate(tqdm(llm_sample.iterrows(), total=len(llm_sample))):
    user_prompt = build_user_prompt(row, include_desc=False)

    raw = call_llm(
        system_prompt, user_prompt,
        api_provider=API_PROVIDER, model=MODEL_NAME, api_key=API_KEY
    )
    llm_raw_responses.append(raw)

    parsed = parse_llm_response(raw)
    llm_predictions.append(parsed['prediction'])
    llm_reasonings.append(parsed['reasoning'])

print(f"\nCompleted: {len(llm_predictions)} predictions")
print(f"Parse errors: {sum(1 for p in llm_predictions if p is None)}")

## Evaluation

In [ ]:
llm_metrics = evaluate_predictions(y_true, llm_predictions, label="LLM (No Desc)")
ann_metrics = evaluate_predictions(y_true, ann_preds.tolist(), label="ANN")

In [ ]:
comparison = compare_results(y_true, llm_predictions, ann_preds.tolist(), llm_reasonings)

print(f"LLM accuracy: {comparison['llm_correct'].mean()*100:.1f}%")
print(f"ANN accuracy: {comparison['ann_correct'].mean()*100:.1f}%")
print(f"\nAgreement between LLM and ANN: {(comparison['llm_pred'] == comparison['ann_pred']).mean()*100:.1f}%")

comparison.head(10)

In [ ]:
# Cases where LLM and ANN disagree
disagree = comparison[comparison['llm_pred'] != comparison['ann_pred']]
print(f"Disagreements: {len(disagree)} / {len(comparison)}")
print(f"LLM correct in disagreements: {disagree['llm_correct'].sum()}")
print(f"ANN correct in disagreements: {disagree['ann_correct'].sum()}")

if len(disagree) > 0:
    print("\nSample disagreements with LLM reasoning:")
    for _, row in disagree.head(5).iterrows():
        actual = 'Fully Paid' if row['actual'] == 1 else 'Charged Off'
        llm = 'Fully Paid' if row['llm_pred'] == 1 else 'Charged Off'
        ann = 'Fully Paid' if row['ann_pred'] == 1 else 'Charged Off'
        print(f"  Actual: {actual} | LLM: {llm} | ANN: {ann}")
        print(f"  Reasoning: {row['llm_reasoning']}\n")

## Export Results

In [ ]:
# Save full results
comparison.to_csv("../../data/processed/04a_llm_no_desc_results.csv", index=False)

# Save summary metrics
summary = pd.DataFrame([llm_metrics, ann_metrics],
                        index=['LLM (No Desc)', 'ANN'])
summary.to_csv("../../data/processed/04a_llm_no_desc_metrics.csv")
print(summary.to_string())